# 01｜只用远端现货生成最终八列表

本 Notebook 只读取一份远端原始现货文件。三状态、非零反转、零段反转、大涨和大跌均由包内冻结代码生成；不读取本地结果，不读取成交或评价记录。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

package_root = Path(os.environ.get('FINAL_UPLOAD_PACKAGE_ROOT', '/home/hzy/cta/20260824_1805_冻结中证500输出包')).expanduser()
if not package_root.is_absolute():
    raise ValueError('FINAL_UPLOAD_PACKAGE_ROOT 必须是绝对路径')
package_root = package_root.resolve()
spot_raw = os.environ.get('COMPANY_SPOT_PATH', '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet')
spot = Path(spot_raw).expanduser()
if not spot.is_absolute():
    raise ValueError('COMPANY_SPOT_PATH 必须是绝对路径')
output = Path(os.environ.get('UPLOAD_OUTPUT_DIR', str(package_root / 'runtime_outputs'))).expanduser()
if not output.is_absolute():
    raise ValueError('UPLOAD_OUTPUT_DIR 必须是绝对路径')
output = output.resolve()
command = [sys.executable, str(package_root / 'src' / 'generate_compact_output.py'), '--spot', str(spot), '--output', str(output)]
subprocess.run(command, cwd=str(package_root), check=True)
print('八列表已生成：', output / '最终执行日简表.csv')

[非零冻结] 启动
[非零退出] 读取冻结 V55；只计算最终 score_02，不重建候选池
[非零退出] V55 冻结信号完成；事件数=45
[非零退出] 读取冻结 V80；只计算最终 score_02，不重建候选池
[非零退出] V80 冻结信号完成；事件数=62
[非零退出] output=/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs/_engine_outputs/remote_nonzero_five_columns.csv
[非零退出] rows=2097 date=2018-01-03 -> 2026-08-25
[零段反转冻结] 启动
[零段反转] 读取唯一远端现货并从冻结八状态公式生成基础状态
[零段反转-down] 运行冻结 V38_down_s01_a1_q0.85_c1_H03（不扫描候选、不读取未来标签）
[零段反转-up] 运行冻结 V57_up_s01_a1_q0.85_c1_H04（不扫描候选、不读取未来标签）
[零段反转] output=/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs/_engine_outputs/remote_zero_transfer_predictions.csv; rows=2097; execution=2018-01-03 -> 2026-08-25
[大跌冻结] 启动
[O2O down 17:59:10] 1/4 读取唯一现货并构造因果特征
[O2O down 17:59:12] 1/4 完成：rows=4766，Development=1213，Validation=482
[O2O down 17:59:12] 2/4 只计算已冻结参数：V156:base_0621_cov_0.075；不重建候选池
[O2O down 17:59:12] 3/4 用固定阈值生成冻结预测，之后才读取 Test
[O2O down 17:59:12] 4/4 冻结参数逐日结果已生成
[大涨大跌-down] prediction_output=/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证5

In [2]:
import pandas as pd
import json
result_path = output / '最终执行日简表.csv'
record_path = output / '最终执行日简表_生成记录.json'
result = pd.read_csv(result_path)
record = json.loads(record_path.read_text(encoding='utf-8'))
expected_columns = ['实际执行日', '三状态', '+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']
assert result.columns.tolist() == expected_columns
assert record['date_mapping']['zero_transfer_formation_to_execution_exact']
display(result.tail(10))
print('执行日范围：', result['实际执行日'].min(), '->', result['实际执行日'].max())
print('最新形成日→执行日：', record['date_mapping']['latest_formation_to_execution'])

,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌
2087,2026-08-12,0,0,0,0,0,0,0
2088,2026-08-13,0,0,0,0,0,0,0
2089,2026-08-14,0,0,0,0,0,0,0
2090,2026-08-17,0,0,0,0,0,0,0
2091,2026-08-18,0,0,0,0,0,0,0
2092,2026-08-19,0,0,0,0,0,0,0
2093,2026-08-20,0,0,0,0,0,0,0
2094,2026-08-21,0,0,0,0,0,0,0
2095,2026-08-24,0,0,0,1,0,0,1
2096,2026-08-25,0,0,0,0,0,0,0


执行日范围： 2018-01-03 -> 2026-08-25
最新形成日→执行日： 2026-08-24 -> 2026-08-25
